# 01 — Design the contract

Design and validate an interface before studying its implementation.

Run this lab in the **dev-platform** workspace on Minikube or Kubernetes, using the `eoepca/pde-code-server:amd64` image. Select its preinstalled **Bash** kernel and use **Shift+Enter**, reading each explanation before executing its cell. No terminal or kernel installation is needed. Stop at an unexpected error and fix it before continuing. The course environment is provisioned by workspace startup.

The kernel keeps environment variables and the working directory between cells. The first cell finds the course checkout and activates its locked environment. Command groups run in a subshell with `set -euo pipefail` so a failing command stops that cell without killing the kernel. Scratch results stay in `.notebook-runs/01-design-the-contract/`; rerunning a cell replaces only its lab outputs.

[Lesson](../modules/01-design-the-contract/lesson.md) · [Exercise](../modules/01-design-the-contract/exercise.md)


## Prepare the notebook workspace

Run this cell after every kernel restart. It clears inherited `CWLTOOL_OPTIONS` so a Coder workspace configured for Podman can still run the explicitly local exercises. It does not change your cluster or publish anything.


In [ ]:
lab_start_dir="$PWD"
while [[ ! -f reference/waterbodies.cwl || ! -f pyproject.toml ]]; do
    if [[ "$PWD" == / ]]; then
        cd "$lab_start_dir"
        break
    fi
    cd ..
done
export WORKSPACE="$PWD"
if [[ -x "$WORKSPACE/.venv/bin/python" && -f "$WORKSPACE/reference/waterbodies.cwl" ]]; then
    source "$WORKSPACE/.venv/bin/activate"
    export PYTHONPATH="$WORKSPACE${PYTHONPATH:+:$PYTHONPATH}"
    export CWLTOOL_OPTIONS=
    export MPLBACKEND=Agg
    export LAB_DIR="$WORKSPACE/.notebook-runs/01-design-the-contract"
    mkdir -p "$LAB_DIR"
    cd "$LAB_DIR"
    python --version
    waterbodies --help
else
    printf 'Open this notebook inside the course checkout. Ask the instructor to check workspace environment provisioning.\n' >&2
    false
fi


## Inspect the canonical interface

CWL `Directory` is a staged catalog, `File` is a raster and `string` carries the AOI and CRS. The `inputBinding` translates an input into a CLI option; `outputBinding.glob` identifies the promised output. Inspect the crop declarations and predict which inputs can be omitted.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
from pathlib import Path
import os
import yaml
contract = yaml.safe_load((Path(os.environ['WORKSPACE']) / 'reference/waterbodies.cwl').read_text())
crop = next(process for process in contract['$graph'] if process['id'] == 'crop')
print(yaml.safe_dump(crop, sort_keys=False))
scratch = {'cwlVersion': contract['cwlVersion'], '$namespaces': contract['$namespaces'], **crop}
Path('scratch-crop.cwl').write_text(yaml.safe_dump(scratch, sort_keys=False))
PYTHON
cwltool --validate scratch-crop.cwl
)


## Make a contract change in scratch space

This cell writes a second tool with a default CRS. Edit the proposed default before running if you want to test another value. Explain why a default changes the behavior of callers that omit a value; CWL validation checks declarations, not the scientific correctness of the CRS choice.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
from pathlib import Path
import yaml
crop = yaml.safe_load(Path('scratch-crop.cwl').read_text())
crop['inputs']['epsg']['default'] = 'EPSG:4326'
Path('crop-with-default.cwl').write_text(yaml.safe_dump(crop, sort_keys=False))
print(yaml.safe_dump(crop['inputs'], sort_keys=False))
PYTHON
cwltool --validate crop-with-default.cwl
)


## Verify the packed graph

The canonical document contains one `Workflow` and four `CommandLineTool` definitions. Its workflow selector is `#waterbodies`. Validation resolves the graph but does not execute it.


In [ ]:
(
set -euo pipefail
cwltool --validate "$WORKSPACE/reference/waterbodies.cwl#waterbodies"
cwltool --validate "$WORKSPACE/modules/11-assure-the-contract/v1/waterbodies.cwl#waterbodies"
cwltool --validate "$WORKSPACE/modules/11-assure-the-contract/v2/waterbodies.cwl#waterbodies"
)


## Record your findings

**Expected outcome:** Your scratch crop tools and all three course graphs validate; you can explain types, bindings, required inputs and defaults.

Add your observations to a Markdown cell: what you predicted, what the command produced, and which contract declaration explains it. Restart the kernel and run all cells with the default settings to check reproducibility.

Compare [the solution](../modules/01-design-the-contract/solution.md) after the activity. Return to [the lab index](README.md).
